In [0]:
%run ../../utils/config.py

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Läs från bronze
df = spark.read.table(BRONZE_TABLE)

# Byt namn på kolumner till snake_case
df = df \
    .withColumnRenamed("Year of event",              "year_of_event") \
    .withColumnRenamed("Event dates",                "event_dates") \
    .withColumnRenamed("Event name",                 "event_name") \
    .withColumnRenamed("Event distance/length",      "event_distance_length") \
    .withColumnRenamed("Event number of finishers",  "event_num_finishers") \
    .withColumnRenamed("Athlete performance",        "athlete_performance") \
    .withColumnRenamed("Athlete country",            "athlete_country") \
    .withColumnRenamed("Athlete year of birth",      "athlete_year_of_birth") \
    .withColumnRenamed("Athlete gender",             "athlete_gender") \
    .withColumnRenamed("Athlete age category",       "athlete_age_category") \
    .withColumnRenamed("Athlete average speed",      "athlete_avg_speed") \
    .withColumnRenamed("Athlete ID",                 "athlete_id_raw")

# Klassificera event-typ
df = df.withColumn(
    "event_unit",
    F.when(F.col("event_distance_length").rlike(r"^\d+(\.\d+)?km"), "km")
     .when(F.col("event_distance_length").rlike(r"^\d+(\.\d+)?mi"), "mi")
     .when(F.col("event_distance_length").rlike(r"^\d+(\.\d+)?h|^\d+:\d+h"), "h")
     .when(F.col("event_distance_length").rlike(r"^\d+(\.\d+)?d"), "d")
     .otherwise("unknown")
)

# Ta bort ogiltiga rader
df = df.filter(~F.col("event_unit").isin("d", "unknown"))

# Validera performance-format
df = df.withColumn(
    "performance_valid",
    F.when(
        F.col("event_unit").isin("km", "mi"),
        F.col("athlete_performance").rlike(r"^\d+:\d{2}:\d{2}\s*h$")
    ).when(
        F.col("event_unit") == "h",
        F.col("athlete_performance").rlike(r"^\d+(\.\d+)?\s*km$")
    ).otherwise(False)
)

df = df.filter(F.col("performance_valid") == True).drop("performance_valid")

# Konvertera performance till rätt datatyp
# km/mi-events: "7:44:06 h" -> sekunder
df = df.withColumn(
    "performance_clean",
    F.trim(F.regexp_replace(F.col("athlete_performance"), r"\s*h$", ""))
)
 
df = df.withColumn(
    "performance_seconds",
    F.when(
        F.col("event_unit").isin("km", "mi"),
        (
            F.split(F.col("performance_clean"), ":")[0].cast("int") * 3600 +
            F.split(F.col("performance_clean"), ":")[1].cast("int") * 60 +
            F.split(F.col("performance_clean"), ":")[2].cast("int")
        )
    ).otherwise(None)
).withColumn(
    # h-events: "84.985 km" -> float
    "performance_km",
    F.when(
        F.col("event_unit") == "h",
        F.trim(F.regexp_replace(F.col("athlete_performance"), r"\s*km$", "")).cast("float")
    ).otherwise(None)
).drop("performance_clean")

# Övrig rensning
df = df.filter(
    F.col("event_name").isNotNull() &
    F.col("athlete_country").isNotNull() &
    F.col("athlete_gender").isNotNull()
)

df = df.withColumn(
    "athlete_approx_age",
    (F.col("year_of_event") - F.col("athlete_year_of_birth").cast("int")).cast("int")
).filter(F.col("athlete_approx_age").between(10, 100))

# Skapa IDs med dense_rank
w_event   = Window.orderBy("event_name")
w_athlete = Window.orderBy("athlete_id_raw")

df = df.withColumn("event_id",   F.dense_rank().over(w_event)) \
       .withColumn("athlete_id", F.dense_rank().over(w_athlete)) \
       .withColumn("result_id",  F.monotonically_increasing_id())

# Välj slutliga kolumner

df_silver = df.select(
    "result_id",
    "event_id",
    "athlete_id",
    "year_of_event",
    "event_dates",
    "event_name",
    "event_distance_length",
    "event_unit",
    "event_num_finishers",
    "athlete_performance",
    "performance_seconds",
    "performance_km",
    "athlete_country",
    "athlete_year_of_birth",
    "athlete_approx_age",
    "athlete_gender",
    "athlete_age_category",
    "athlete_avg_speed",
    "_ingested_at",
)

# Skriv till silver Delta-tabell
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f" Silver-tabell skapad: {SILVER_TABLE}")
spark.sql(f"SELECT count(*) as antal_rader FROM {SILVER_TABLE}").display()

df_silver.limit(10).display()